# Endometrial Cancer (EC) Cohort Definition — All of Us CDR v9

Builds the EC case/control cohort used for the PRS analysis, from the full AoU CDR v9 population through to the final analysis cohort (2,386 cases / 269,416 controls, N = 271,802).

In [ ]:
import os
import pandas as pd
import numpy as np
import pandas_gbq
import subprocess
from datetime import datetime


In [ ]:
# Loading workspace variables (dataset, bucket_output, etc.)
%run /home/jupyter/Var_set_2.ipynb
project_id = os.environ.get('GOOGLE_PROJECT', '').strip("'\"")
CDR = os.environ['WORKSPACE_CDR']
print(project_id)


## Case identification

In [ ]:
# Pulling all EC records via ICD-10 C54.* (malignant neoplasm of corpus uteri, incl. endometrium) and its ICD-9 182.* legacy equivalent
query = f"""
SELECT
    co.person_id,
    co.condition_occurrence_id,
    c.concept_code,
    c.vocabulary_id,
    co.condition_start_date,
    co.condition_source_concept_id
FROM `{dataset}.condition_occurrence` co
JOIN `{dataset}.concept` c
    ON c.concept_id = co.condition_source_concept_id
WHERE (c.vocabulary_id = 'ICD10CM' AND c.concept_code LIKE 'C54%')
   OR (c.vocabulary_id = 'ICD9CM' AND c.concept_code LIKE '182%')
"""
ec_all_records = pandas_gbq.read_gbq(query, project_id=project_id, dialect="standard")
print(f"Total records: {len(ec_all_records):,}")
print(f"Unique people: {ec_all_records['person_id'].nunique():,}")


In [ ]:
# Collapsing to one row per person
ec_person = (
    ec_all_records
    .groupby('person_id')
    .agg(
        first_diagnosis_date=('condition_start_date', 'min'),
        last_diagnosis_date=('condition_start_date', 'max'),
        n_records=('condition_start_date', 'count'),
        codes_seen=('concept_code', lambda x: sorted(set(x)))
    )
    .reset_index()
)
print(f"Total unique EC cases via ICD (person-level): {len(ec_person):,}")


In [ ]:
# Some EC diagnoses are coded only via SNOMED concepts (e.g. "Primary malignant neoplasm of uterus"), with no corresponding C54 ICD code
# This curated list closes that coding gap without loosening the case definition

snomed_include_ids = [
    # direct EC/endometrial malignancy concepts
    4110871,   # 254878006          Endometrial carcinoma
    4095749,   # 188192002          Malignant neoplasm of endometrium of corpus uteri
    4247238,   # 93781006           Primary malignant neoplasm of endometrium
    4092518,   # 188194001          Malignant neoplasm of corpus uteri NOS
    3568073,   # 533381000000106    Malignant neoplasm of corpus uteri NOS
    3654964,   # 840516000          Complex mixed and stromal malignant neoplasm of endometrium of corpus uteri
    37016124,  # 107771000119106    Primary malignant clear cell neoplasm of endometrium
    37016123,  # 107751000119102    Primary malignant mixed Mullerian neoplasm of endometrium

    # uterine-corpus/body malignancy concepts
    45770892,  # 10708511000119108  Primary malignant neoplasm of uterus
    197230,    # 371973000          Malignant neoplasm of uterus
    4048226,   # 123845008          Adenocarcinoma of endometrium
    4156113,   # 371972005          Malignant neoplasm of body of uterus
    4095748,   # 188191009          Malignant neoplasm of fundus of corpus uteri
    194286,    # 188189001          Malignant neoplasm of corpus uteri, excluding isthmus
    4162860,   # 371971003          Primary malignant neoplasm of body of uterus
    4091474,   # 188193007          Malignant neoplasm of myometrium of corpus uteri
    4095747,   # 188190005          Malignant neoplasm of cornu of corpus uteri
    198082,    # 109879008          Overlapping malignant neoplasm of body of uterus

    # histology-specific primary EC concepts
    40493028,  # 449073009          Carcinoma of corpus uteri
    3654966,   # 840518004          Complex mixed and stromal malignant neoplasm of corpus uteri
    3654961,   # 840513008          Complex mixed and stromal malignant neoplasm of myometrium of corpus uteri
    37166788,  # 1260086007         Endometrioid carcinoma of endometrium
    36676728,  # 773774000          High-grade neuroendocrine carcinoma of corpus uteri
    36676409,  # 773284000          Malignant germ cell neoplasm of corpus uteri
    37016125,  # 107791000119107    Primary adenosquamous carcinoma of endometrium
    42538693,  # 762458004          Primary endometrioid carcinoma of endometrium
    1245882,   # 16824021000119104  Primary malignant neuroendocrine neoplasm of uterus
    37166810,  # 1260116000         Primary malignant stromal sarcoma of endometrium
    36716615,  # 722681008          Primary mixed adenocarcinoma of endometrium
    36716614,  # 722679006          Primary mucinous adenocarcinoma of endometrium
    37162696,  # 1197260003         Primary poorly differentiated endocrine carcinoma of corpus uteri
    36717227,  # 722680009          Primary serous adenocarcinoma of endometrium
    36716616,  # 722682001          Primary small cell neuroendocrine carcinoma of endometrium
    37116596,  # 733359005          Primary squamous cell carcinoma of endometrium
    37167599,  # 1268545005         Primary transitional cell carcinoma of corpus uteri
    37116597,  # 733360000          Primary undifferentiated carcinoma of endometrium
    35608092,  # 764737005          Squamous cell carcinoma of corpus uteri
    37203941,  # 785807007          Transitional cell carcinoma of corpus uteri
    35624158,  # 766758001          Undifferentiated carcinoma of corpus uteri
    37165674,  # 1255394008         Undifferentiated carcinoma of endometrium
    603323,    # 16754641000119104  Abnormal uterine bleeding due to primary malignant neoplasm of endometrium
]

print(f"Total SNOMED concepts in case definition: {len(snomed_include_ids)}")
print(f"Unique (no duplicates): {len(set(snomed_include_ids))}")


In [ ]:
# Checking how many people the SNOMED concepts add beyond the ICD-based case list
snomed_in = "(" + ",".join(str(c) for c in snomed_include_ids) + ")"

query = f"""
SELECT DISTINCT co.person_id
FROM `{dataset}.condition_occurrence` co
WHERE co.condition_concept_id IN {snomed_in}
   OR co.condition_source_concept_id IN {snomed_in}
"""
snomed_persons = pandas_gbq.read_gbq(query, project_id=project_id, dialect="standard")
snomed_set = set(snomed_persons['person_id'])
icd_set = set(ec_person['person_id'])

print(f"People via SNOMED: {len(snomed_set):,}")
print(f"People via ICD (already have): {len(icd_set):,}")
print(f"New people SNOMED adds beyond ICD: {len(snomed_set - icd_set):,}")


In [ ]:
# Building the complete case list by adding the SNOMED-only people to the ICD-based case list
snomed_only_ids = snomed_set - icd_set

snomed_dates_query = f"""
SELECT co.person_id, co.condition_start_date
FROM `{dataset}.condition_occurrence` co
WHERE (co.condition_concept_id IN {snomed_in} OR co.condition_source_concept_id IN {snomed_in})
  AND co.person_id IN {tuple(int(x) for x in snomed_only_ids)}
"""
snomed_only_records = pandas_gbq.read_gbq(snomed_dates_query, project_id=project_id, dialect="standard")

snomed_only_person = (
    snomed_only_records
    .groupby('person_id')
    .agg(first_diagnosis_date=('condition_start_date', 'min'),
         last_diagnosis_date=('condition_start_date', 'max'),
         n_records=('condition_start_date', 'count'))
    .reset_index()
)
snomed_only_person['codes_seen'] = [['SNOMED_only']] * len(snomed_only_person)

ec_person_full = pd.concat([ec_person, snomed_only_person], ignore_index=True)
print(f"EC person list before SNOMED: {len(ec_person):,}")
print(f"EC person list after SNOMED:  {len(ec_person_full):,}  (+{len(ec_person_full) - len(ec_person)})")


In [ ]:
# Recomputing first/last_diagnosis_date and n_records using the earliest/latest qualifying date across both ICD and SNOMED evidence
# Doing this for every case (not just SNOMED-only people who were already correctly combined above since by definition they have no ICD evidence).
all_case_ids = tuple(int(x) for x in ec_person_full['person_id'])

snomed_dates_all_query = f"""
SELECT co.person_id, co.condition_occurrence_id, co.condition_start_date
FROM `{dataset}.condition_occurrence` co
WHERE (co.condition_concept_id IN {snomed_in} OR co.condition_source_concept_id IN {snomed_in})
  AND co.person_id IN {all_case_ids}
"""
snomed_dates_all = pandas_gbq.read_gbq(snomed_dates_all_query, project_id=project_id, dialect="standard")
snomed_dates_all['condition_start_date'] = pd.to_datetime(snomed_dates_all['condition_start_date'])

snomed_first_by_person = (
    snomed_dates_all.groupby('person_id')['condition_start_date']
    .min()
    .rename('snomed_first_date')
    .reset_index()
)

ec_person_full['first_diagnosis_date'] = pd.to_datetime(ec_person_full['first_diagnosis_date'])
ec_person_full_check = ec_person_full.merge(snomed_first_by_person, on='person_id', how='left')

# Earliest qualifying date across both systems
ec_person_full_check['corrected_first_diagnosis_date'] = ec_person_full_check[
    ['first_diagnosis_date', 'snomed_first_date']
].min(axis=1)

moved_mask = (
    ec_person_full_check['snomed_first_date'].notna() &
    (ec_person_full_check['snomed_first_date'] < ec_person_full_check['first_diagnosis_date'])
)
n_moved = moved_mask.sum()
days_moved = (
    ec_person_full_check.loc[moved_mask, 'first_diagnosis_date']
    - ec_person_full_check.loc[moved_mask, 'snomed_first_date']
).dt.days

print(f"People whose first_diagnosis_date moves earlier after this correction: {n_moved:,} / {len(ec_person_full_check):,}")
if n_moved > 0:
    print(f"  Median days moved earlier: {days_moved.median():.0f}")
    print(f"  Max days moved earlier: {days_moved.max():.0f}")

ec_person_full['first_diagnosis_date'] = ec_person_full_check['corrected_first_diagnosis_date']
print(f"\nfirst_diagnosis_date corrected for all {len(ec_person_full):,} cases using earliest-across-ICD-and-SNOMED logic.")

# Recombining last_diagnosis_date and n_records across both ICD and SNOMED evidence for everyone, matching the fix just applied to first_diagnosis_date.
icd_only_dates = (
    ec_all_records.groupby('person_id')
    .agg(icd_first=('condition_start_date', 'min'),
         icd_last=('condition_start_date', 'max'))
    .reset_index()
)
icd_occurrence_ids = ec_all_records.groupby('person_id')['condition_occurrence_id'].apply(set)
icd_only_dates['icd_first'] = pd.to_datetime(icd_only_dates['icd_first'])
icd_only_dates['icd_last'] = pd.to_datetime(icd_only_dates['icd_last'])

snomed_last_n_by_person = (
    snomed_dates_all.groupby('person_id')['condition_start_date']
    .agg(['max'])
    .rename(columns={'max': 'snomed_last'})
    .reset_index()
)
snomed_occurrence_ids = snomed_dates_all.groupby('person_id')['condition_occurrence_id'].apply(set)

meta_check = ec_person_full[['person_id']].merge(icd_only_dates, on='person_id', how='left')
meta_check = meta_check.merge(snomed_last_n_by_person, on='person_id', how='left')

meta_check['combined_last_diagnosis_date'] = meta_check[['icd_last', 'snomed_last']].max(axis=1)

# Deduplicate by condition_occurrence_id across the ICD and SNOMED record sets, so a row matching both filters is counted once, not twice.
occurrence_union = {}
for pid, ids in icd_occurrence_ids.items():
    occurrence_union[pid] = set(ids)
for pid, ids in snomed_occurrence_ids.items():
    occurrence_union.setdefault(pid, set()).update(ids)
combined_n_records_map = {pid: len(ids) for pid, ids in occurrence_union.items()}
meta_check['combined_n_records'] = meta_check['person_id'].map(combined_n_records_map).fillna(0).astype(int)

n_last_date_changed = (
    meta_check['combined_last_diagnosis_date'].notna() &
    meta_check['icd_last'].notna() &
    (meta_check['combined_last_diagnosis_date'] > meta_check['icd_last'])
).sum()
print(f"People whose last_diagnosis_date moves later once SNOMED evidence is combined in: {n_last_date_changed:,}")

ec_person_full = ec_person_full.merge(
    meta_check[['person_id', 'combined_last_diagnosis_date', 'combined_n_records']],
    on='person_id', how='left'
)
ec_person_full['last_diagnosis_date'] = ec_person_full['combined_last_diagnosis_date']
ec_person_full['n_records'] = ec_person_full['combined_n_records']
ec_person_full = ec_person_full.drop(columns=['combined_last_diagnosis_date', 'combined_n_records'])
print(f"last_diagnosis_date and n_records recombined across ICD+SNOMED evidence for all {len(ec_person_full):,} cases.")


## Base cohort: WGS, ancestry, genetic sex

In [ ]:
# Loading the ancestry predictions file
ancestry_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv"

ancestry_pred = pd.read_csv(
    ancestry_path,
    delimiter="\t",
    storage_options={"requester_pays": True, "project": project_id}
)
print(f"WGS participants: {len(ancestry_pred):,}")


In [ ]:
# Extracting principal components and builing the WGS base dataframe. 
# The ancestry file's stricter classification (low-confidence calls assigned to an explicit "Other/unclassified" group is adopted as ancestry_pred
# The forced 6-group assignment (originally ancestry_pred) is kept as ancestry_pred_forced.
PCs = ancestry_pred["pca_features"].str[1:-1].str.split(",", expand=True).astype(float)
PCs.columns = [f"PC{i}" for i in range(1, PCs.shape[1]+1)]

wgs_df = pd.concat(
    [ancestry_pred[["research_id", "ancestry_pred", "ancestry_pred_other"]], PCs],
    axis=1
).rename(columns={"research_id": "person_id"})

wgs_df = wgs_df.rename(columns={"ancestry_pred": "ancestry_pred_forced"})
wgs_df = wgs_df.rename(columns={"ancestry_pred_other": "ancestry_pred"})

print(f"Number of PCs extracted: {PCs.shape[1]}")
print(f"\nancestry_pred (strict) distribution:")
print(wgs_df['ancestry_pred'].value_counts(dropna=False))


In [ ]:
# Loading genetic sex (dragen_sex_ploidy) and merging onto the WGS cohort
genomics_metrics_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/qc/genomics_metrics.tsv"

genomics_metrics = pd.read_csv(
    genomics_metrics_path,
    delimiter="\t",
    storage_options={"requester_pays": True, "project": project_id}
)

sex_df = genomics_metrics[['research_id', 'dragen_sex_ploidy']].rename(columns={'research_id': 'person_id'})
wgs_df = wgs_df.merge(sex_df, on='person_id', how='left')
print(wgs_df['dragen_sex_ploidy'].value_counts(dropna=False))


In [ ]:
# Restricting to genetic females (XX), assigning the EC_case flag
wgs_df['EC_case'] = wgs_df['person_id'].isin(ec_person_full['person_id']).astype(int)
wgs_female = wgs_df[wgs_df['dragen_sex_ploidy'] == 'XX'].copy()

print(f"Genetic females (XX) with WGS: {len(wgs_female):,}")
print(wgs_female['EC_case'].value_counts())


## Exclusions

In [ ]:
# Controls with a prior hysterectomy (removing the organ at risk) or a prior cancer diagnosis other than non-melanoma skin cancer (C44) are excluded from the control pool.
# Finding all cancer ICD-10 concept IDs (excluding non-melanoma skin cancer, C44) to identify controls with prior malignancy
query_cancer_concepts = f"""
SELECT DISTINCT
    concept_id,
    concept_code,
    concept_name
FROM `{dataset}.concept`
WHERE vocabulary_id = 'ICD10CM'
  AND concept_code LIKE 'C%'
  AND concept_code NOT LIKE 'C44%'
  AND SAFE_CAST(REGEXP_EXTRACT(concept_code, r'^C(\d+)') AS INT64) BETWEEN 0 AND 97
"""
cancer_concepts = pandas_gbq.read_gbq(query_cancer_concepts, project_id=project_id, dialect="standard")
print(f"Cancer concept IDs found (excluding C44 non-melanoma skin cancer): {len(cancer_concepts)}")


In [ ]:
# Finding controls with a prior cancer diagnosis
control_ids = wgs_female[wgs_female['EC_case'] == 0]['person_id'].tolist()
cancer_concept_ids = tuple(cancer_concepts['concept_id'].astype(int).tolist())

query_controls_cancer = f"""
SELECT DISTINCT person_id
FROM `{dataset}.condition_occurrence`
WHERE condition_source_concept_id IN {cancer_concept_ids}
"""
anyone_with_cancer = pandas_gbq.read_gbq(query_controls_cancer, project_id=project_id, dialect="standard")

cancer_person_ids = set(int(x) for x in anyone_with_cancer['person_id'])
control_ids_set = set(int(x) for x in control_ids)
controls_with_cancer = control_ids_set & cancer_person_ids

print(f"Total controls (pre-exclusion): {len(control_ids_set):,}")
print(f"Controls with a prior cancer diagnosis: {len(controls_with_cancer):,}")


In [ ]:
# Finding hysterectomy procedure records for the whole cohort (used for both control exclusion and case-timing checks)
hyst_concept_ids_query = f"""
SELECT c.concept_id, c.concept_code, c.concept_name, c.vocabulary_id
FROM `{dataset}.concept` c
JOIN `{dataset}.procedure_occurrence` po
    ON c.concept_id = po.procedure_source_concept_id
WHERE LOWER(c.concept_name) LIKE '%hysterectomy%'
GROUP BY c.concept_id, c.concept_code, c.concept_name, c.vocabulary_id
"""
hyst_concept_ids = pandas_gbq.read_gbq(hyst_concept_ids_query, project_id=project_id, dialect="standard")
id_list = tuple(hyst_concept_ids['concept_id'].astype(int).tolist())

hyst_query = f"""
SELECT person_id, procedure_source_concept_id, procedure_date
FROM `{dataset}.procedure_occurrence`
WHERE procedure_source_concept_id IN {id_list}
"""
hyst_records = pandas_gbq.read_gbq(hyst_query, project_id=project_id, dialect="standard")

hyst_person = (
    hyst_records
    .groupby('person_id')
    .agg(first_hyst_date=('procedure_date', 'min'), n_hyst_records=('procedure_date', 'count'))
    .reset_index()
)
print(f"People with a hysterectomy record: {len(hyst_person):,}")


In [ ]:
# Excluding controls with a hysterectomy record
hyst_ids_set = set(int(x) for x in hyst_person['person_id'])
controls_with_hyst = control_ids_set & hyst_ids_set

print(f"Controls with a hysterectomy record: {len(controls_with_hyst):,}")
print(f"Controls with prior cancer OR hysterectomy: {len(controls_with_cancer | controls_with_hyst):,}")
print(f"Clean controls remaining: {len(control_ids_set - (controls_with_cancer | controls_with_hyst)):,}")


In [ ]:
# For the controls excluded by prior cancer or hysterectomy, checking whether the qualifying event fell before, on, or after consent
# Descriptive only (controls are excluded if the record exists at any time)
controls_excluded_ids = controls_with_cancer | controls_with_hyst

consent_query_check = f"""
SELECT person_id, MIN(observation_date) AS consent_date
FROM `{CDR}.observation`
WHERE person_id IN ({','.join(str(i) for i in controls_excluded_ids)})
  AND observation_source_concept_id = 1586099
GROUP BY person_id
"""
consent_check = pandas_gbq.read_gbq(consent_query_check, project_id=project_id, dialect="standard")
consent_check['consent_date'] = pd.to_datetime(consent_check['consent_date'])
consent_map = consent_check.set_index('person_id')['consent_date']

def summarise_timing(excl_ids, dates_df, date_col, label):
    dates_df = dates_df.copy()
    dates_df['consent_date'] = dates_df['person_id'].map(consent_map)
    n_total = len(excl_ids)
    n_no_date_record = n_total - len(dates_df)  # excluded but no qualifying event date found (shouldn't normally happen)
    n_missing_consent = dates_df['consent_date'].isna().sum()
    has_both = dates_df.dropna(subset=['consent_date', date_col])
    n_before = (has_both[date_col] < has_both['consent_date']).sum()
    n_on = (has_both[date_col] == has_both['consent_date']).sum()
    n_after = (has_both[date_col] > has_both['consent_date']).sum()

    print(f"{label}: {n_total:,} controls excluded")
    print(f"  Event before consent:  {n_before:,} ({100*n_before/n_total:.1f}% of total)")
    print(f"  Event on consent date: {n_on:,} ({100*n_on/n_total:.1f}% of total)")
    print(f"  Event after consent:   {n_after:,} ({100*n_after/n_total:.1f}% of total)")
    print(f"  Missing consent date:  {n_missing_consent:,} ({100*n_missing_consent/n_total:.1f}% of total)")
    if n_no_date_record > 0:
        print(f"  No qualifying event date found: {n_no_date_record:,} (unexpected -- investigate)")
    print()

# Cancer exclusion timing
cancer_excl_ids = controls_with_cancer
cancer_date_query = f"""
SELECT person_id, MIN(condition_start_date) AS first_cancer_date
FROM `{dataset}.condition_occurrence`
WHERE condition_source_concept_id IN {cancer_concept_ids}
  AND person_id IN ({','.join(str(i) for i in cancer_excl_ids)})
GROUP BY person_id
"""
cancer_dates = pandas_gbq.read_gbq(cancer_date_query, project_id=project_id, dialect="standard")
cancer_dates['first_cancer_date'] = pd.to_datetime(cancer_dates['first_cancer_date'])
summarise_timing(cancer_excl_ids, cancer_dates, 'first_cancer_date', 'Controls excluded for prior cancer')

# Hysterectomy exclusion timing
hyst_excl_ids = controls_with_hyst
hyst_date_query = f"""
SELECT person_id, MIN(procedure_date) AS first_hyst_date_check
FROM `{dataset}.procedure_occurrence`
WHERE procedure_source_concept_id IN {id_list}
  AND person_id IN ({','.join(str(i) for i in hyst_excl_ids)})
GROUP BY person_id
"""
hyst_dates_check = pandas_gbq.read_gbq(hyst_date_query, project_id=project_id, dialect="standard")
hyst_dates_check['first_hyst_date_check'] = pd.to_datetime(hyst_dates_check['first_hyst_date_check'])
summarise_timing(hyst_excl_ids, hyst_dates_check, 'first_hyst_date_check', 'Controls excluded for hysterectomy')


In [ ]:
# Related individuals are removed using AoU's precomputed pairwise kinship file 
# (threshold kin > 0.0884, approximately 2nd-degree relatedness or closer)
# Where a related pair spans a case and a control, the control is removed to preserve case count.

# Loading the kinship (relatedness) file
kinship_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/relatedness/samples_relatedness.tsv"

kinship = pd.read_csv(
    kinship_path,
    delimiter="\t",
    storage_options={"requester_pays": True, "project": project_id}
)
print(f"Kinship pairs (raw file): {len(kinship):,}")


In [ ]:
# Restricting kinship pairs to our cohort, applying the 2nd-degree relatedness threshold (kin > 0.0884)
all_analysis_ids = set(int(pid) for pid in wgs_female['person_id'])

kinship_in_cohort = kinship[
    (kinship['i.s'].isin(all_analysis_ids)) &
    (kinship['j.s'].isin(all_analysis_ids))
].copy()

related_pairs = kinship_in_cohort[kinship_in_cohort['kin'] > 0.0884].copy()

print(f"Kinship pairs in our cohort: {len(kinship_in_cohort):,}")
print(f"Related pairs (2nd degree or closer): {len(related_pairs):,}")


In [ ]:
# For each related pair, marking one person for removal, prioritising keeping cases over controls
ec_case_ids_current = set(int(pid) for pid in wgs_female.loc[wgs_female['EC_case']==1, 'person_id'])

to_remove = set()
for _, row in related_pairs.iterrows():
    id1, id2 = int(row['i.s']), int(row['j.s'])
    if id1 in to_remove or id2 in to_remove:
        continue
    is_case_1 = id1 in ec_case_ids_current
    is_case_2 = id2 in ec_case_ids_current
    if is_case_1 and not is_case_2:
        to_remove.add(id2)
    elif is_case_2 and not is_case_1:
        to_remove.add(id1)
    else:
        to_remove.add(id2)

n_remove = len(to_remove)
cases_removed = sum(1 for pid in to_remove if pid in ec_case_ids_current)
controls_removed = n_remove - cases_removed

print("Individuals to remove (one from each related pair):", n_remove)
print("Cases to remove:", cases_removed)
print("Controls to remove:", controls_removed)

# Preserving relatedness-only counts under distinct names before the QC-flag step (below) merges into to_remove
n_remove_related_only = n_remove
cases_removed_related_only = cases_removed
controls_removed_related_only = controls_removed


In [ ]:
# Excluding AoU genotype-QC-flagged samples (flagged_samples.tsv) 
# Someone is "flagged" if they fail at least one residual QC metric
flagged_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv"

flagged_samples = pd.read_csv(
    flagged_path,
    delimiter="\t",
    storage_options={"requester_pays": True, "project": project_id}
)

fail_cols = [c for c in flagged_samples.columns if c.startswith('fail_') and c.endswith('_residual')]
is_flagged = flagged_samples[fail_cols].any(axis=1)

flagged_ids = set(int(x) for x in flagged_samples.loc[is_flagged, 's'])

flagged_in_cohort = flagged_ids & all_analysis_ids
flagged_cases = flagged_in_cohort & ec_case_ids_current
flagged_controls = flagged_in_cohort - flagged_cases

print(f"Flagged AoU-wide: {len(flagged_ids):,}")
print(f"Flagged, in our cohort: {len(flagged_in_cohort):,}")
print(f"  of which cases: {len(flagged_cases):,}")
print(f"  of which controls: {len(flagged_controls):,}")

to_remove |= flagged_in_cohort
cases_removed = sum(1 for pid in to_remove if pid in ec_case_ids_current)
controls_removed = len(to_remove) - cases_removed

print(f"\nUpdated to_remove total (related + flagged): {len(to_remove):,}")
print(f"  Cases to remove (cumulative): {cases_removed:,}")
print(f"  Controls to remove (cumulative): {controls_removed:,}")

# QC-flag-only counts (the increment this step adds on top of relatedness removal)
cases_removed_qc_only = len(flagged_cases)
controls_removed_qc_only = len(flagged_controls)


In [ ]:
# EC cases with a hysterectomy recorded more than 1 year before the EC diagnosis date are excluded
# This timing is inconsistent with the hysterectomy being diagnostic or therapeutic for the EC, and more likely reflects a data-quality issue.

# Restricting EC case list to WGS + genetic female
ec_wgs_female = ec_person_full.merge(
    wgs_female[wgs_female['EC_case'] == 1][['person_id']],
    on='person_id', how='inner'
)

In [ ]:
# Classifying hysterectomy timing relative to diagnosis
def hyst_timing(days):
    if pd.isna(days):
        return 'No hysterectomy'
    elif days > 365:
        return 'Major timing issue (>1yr before diagnosis)'
    elif days > 0:
        return 'Before diagnosis (<=1yr)'
    elif days == 0:
        return 'Same day'
    else:
        return 'After diagnosis (treatment)'

ec_hyst = ec_wgs_female.merge(hyst_person, on='person_id', how='left')
ec_hyst['first_diagnosis_date'] = pd.to_datetime(ec_hyst['first_diagnosis_date'])
ec_hyst['first_hyst_date'] = pd.to_datetime(ec_hyst['first_hyst_date'])
ec_hyst['days_hyst_after_diagnosis'] = (ec_hyst['first_hyst_date'] - ec_hyst['first_diagnosis_date']).dt.days
ec_hyst['hyst_timing'] = (-ec_hyst['days_hyst_after_diagnosis']).apply(hyst_timing)

print(f"Total EC cases (WGS + female): {len(ec_hyst):,}")
print(ec_hyst['hyst_timing'].value_counts())


In [ ]:
# Applying the exclusion
ec_after_hyst_exclusion = ec_hyst[ec_hyst['hyst_timing'] != 'Major timing issue (>1yr before diagnosis)'].copy()
print(f"Excluded (hysterectomy >1yr before diagnosis): {len(ec_hyst) - len(ec_after_hyst_exclusion):,}")
print(f"EC cases after hysterectomy-timing exclusion: {len(ec_after_hyst_exclusion):,}")


## Assembling combined cohort

In [ ]:
# Assembling final cases and controls, removing related/QC-flagged individuals
final_cases = ec_after_hyst_exclusion[~ec_after_hyst_exclusion['person_id'].isin(to_remove)].copy()

controls_excluded = controls_with_cancer | controls_with_hyst | to_remove
final_controls_ids = control_ids_set - controls_excluded
final_controls = wgs_female[
    (wgs_female['EC_case'] == 0) & (wgs_female['person_id'].isin(final_controls_ids))
].copy()

print(f"Final EC cases: {len(final_cases):,}")
print(f"Final controls: {len(final_controls):,}")
print(f"Case:control ratio 1:{len(final_controls)/len(final_cases):.1f}")


In [ ]:
# Combining final cases and controls into one cohort dataframe with a shared column set and EC_case flag
final_cases['single_record_flag'] = final_cases['n_records'] == 1

ancestry_cols = ['person_id', 'ancestry_pred', 'ancestry_pred_forced'] + [f'PC{i}' for i in range(1, 17)] + ['dragen_sex_ploidy']
final_cases_full = final_cases.merge(wgs_female[ancestry_cols], on='person_id', how='left')
final_cases_full['EC_case'] = 1

case_cols = ['person_id', 'EC_case', 'first_diagnosis_date', 'last_diagnosis_date', 'n_records',
             'single_record_flag', 'hyst_timing', 'ancestry_pred', 'ancestry_pred_forced',
             'dragen_sex_ploidy'] + [f'PC{i}' for i in range(1, 17)]
control_cols = ['person_id', 'EC_case', 'ancestry_pred', 'ancestry_pred_forced',
                'dragen_sex_ploidy'] + [f'PC{i}' for i in range(1, 17)]

cases_out = final_cases_full[case_cols].copy()
controls_out = final_controls[control_cols].copy()
controls_out['first_diagnosis_date'] = pd.NaT
controls_out['last_diagnosis_date'] = pd.NaT
controls_out['n_records'] = 0
controls_out['single_record_flag'] = False
controls_out['hyst_timing'] = 'No EC (control)'

final_cohort = pd.concat([cases_out, controls_out], ignore_index=True)

print(f"Combined cohort (before self-report/treatment updates): {len(final_cohort):,}")
print(final_cohort['EC_case'].value_counts())


## Self-reported EC and treatment-identified cases

In [ ]:
# Controls who self-report personal EC on the AoU survey (concept 43528679) but have no matching EHR diagnosis record are checked for a plausibly confusable gynaecological diagnosis
# This is for documentation only
# Self-report of a specific cancer diagnosis is a strong enough signal to warrant exclusion regardless of a plausible benign alternative, so all self-reporters are excluded from controls regardless of the outcome of that check.

# Querying controls who self-report personal endometrial cancer with no matching EHR record
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

control_ids = final_cohort.loc[final_cohort['EC_case']==0, 'person_id'].astype(int).tolist()

parts = []
for batch in chunks(control_ids, 40000):
    ids = ','.join(map(str, batch))
    q = f"""
    SELECT DISTINCT o.person_id
    FROM `{CDR}.observation` o
    WHERE o.person_id IN ({ids})
      AND (
        o.observation_concept_id = 43528679
        OR o.value_source_concept_id = 43528679
        OR o.value_as_concept_id = 43528679
        OR o.observation_source_concept_id = 43528679
      )
    """
    parts.append(pandas_gbq.read_gbq(q, project_id=project_id, dialect='standard', progress_bar_type=None))

selfrep = pd.concat(parts, ignore_index=True).drop_duplicates()
selfrep_ids = sorted(set(selfrep['person_id'].astype(int)))
print(f"Controls self-reporting personal endometrial cancer: {len(selfrep_ids):,}")


In [ ]:
# Checking each self-reporter for a plausibly confusable gynaecological diagnosis 
# Documentation only, all self-reporters are excluded regardless of the outcome of this check
ids_str = ','.join(map(str, selfrep_ids))

confusable_patterns = {
    'uterine_fibroid':         ["%leiomyoma of uterus%", "%uterine fibroid%", "%fibroid%uterus%"],
    'endometriosis':           ["%endometriosis%"],
    'endometrial_hyperplasia': ["%endometrial hyperplasia%", "%hyperplasia of endometrium%",
                                 "%complex atypical hyperplasia%", "%atypical endometrial hyperplasia%"],
    'endometrial_polyp':       ["%endometrial polyp%", "%polyp of endometrium%", "%uterine polyp%"],
    'abnormal_uterine_bleed':  ["%abnormal uterine bleeding%", "%postmenopausal bleeding%",
                                 "%menometrorrhagia%", "%dysfunctional uterine bleeding%"],
    'endometrial_intraepithelial_neoplasia': ["%endometrial intraepithelial neoplasia%"],
}

like_clauses = [f"LOWER(concept_name) LIKE '{p}'" for pats in confusable_patterns.values() for p in pats]
concept_q = f"""
SELECT concept_id, concept_name
FROM `{CDR}.concept`
WHERE domain_id = 'Condition' AND ({' OR '.join(like_clauses)})
"""
conf_concepts = pandas_gbq.read_gbq(concept_q, project_id=project_id, dialect='standard', progress_bar_type=None)
conf_ids_str = ','.join(map(str, conf_concepts['concept_id'].tolist()))

q = f"""
SELECT DISTINCT co.person_id
FROM `{CDR}.condition_occurrence` co
WHERE co.person_id IN ({ids_str})
  AND (co.condition_concept_id IN ({conf_ids_str}) OR co.condition_source_concept_id IN ({conf_ids_str}))
"""
has_confusable = pandas_gbq.read_gbq(q, project_id=project_id, dialect='standard', progress_bar_type=None)
has_confusable_ids = set(has_confusable['person_id'].astype(int))

print(f"Self-reporters WITH a plausibly confusable diagnosis: {len(has_confusable_ids):,}")
print(f"Self-reporters with NO confusable diagnosis:          {len(selfrep_ids) - len(has_confusable_ids):,}")


In [ ]:
# Applying the exclusion by dropping ALL self-reporting controls from the cohort
before_ctrl = int((final_cohort['EC_case']==0).sum())

final_cohort = final_cohort[~final_cohort['person_id'].isin(selfrep_ids)].copy()

print(f"Self-reporting controls excluded: {len(selfrep_ids):,}")
print(f"Controls: {before_ctrl:,} -> {int((final_cohort['EC_case']==0).sum()):,}")


In [ ]:
# Selection-order risk check -- selfrep_ids only covers self-reporters who survived the earlier generic hysterectomy exclusion
# This also matches the EC-specific staging surgery this treatment-identified-case query looks for
# A qualifying self-reporter could in principle already be gone before reaching selfrep_ids. 
# Checking against the full pre-exclusion control pool (wgs_female), not selfrep_ids itself, so the check can actually catch this.
pre_excl_control_ids = wgs_female.loc[wgs_female['EC_case'] == 0, 'person_id'].astype(int).tolist()
parts_pre = []
for batch in chunks(pre_excl_control_ids, 40000):
    ids = ','.join(map(str, batch))
    q = f"""
    SELECT DISTINCT o.person_id
    FROM `{CDR}.observation` o
    WHERE o.person_id IN ({ids})
      AND ( o.observation_concept_id = 43528679 OR o.value_source_concept_id = 43528679
         OR o.value_as_concept_id = 43528679 OR o.observation_source_concept_id = 43528679 )
    """
    parts_pre.append(pandas_gbq.read_gbq(q, project_id=project_id, dialect='standard', progress_bar_type=None))
selfrep_ids_pre_excl = set(pd.concat(parts_pre, ignore_index=True)['person_id'].astype(int))

selfreporters_removed_by_hyst_rule = selfrep_ids_pre_excl & controls_with_hyst
if selfreporters_removed_by_hyst_rule:
    print(f"WARNING: {len(selfreporters_removed_by_hyst_rule)} self-reporter(s) were removed by the "
          f"earlier generic hysterectomy-exclusion rule before reaching selfrep_ids -- they are NOT "
          f"visible to the treatment-identified-case query below. Check manually whether any has a "
          f"qualifying EC-specific procedure and no competing cancer code.")
else:
    print("No self-reporter was removed by the earlier generic hysterectomy-exclusion rule -- "
          "the treatment-identified-case query below sees the full self-reporting population.")

# Treatment-identified cases: self-reporters (just excluded above) with an EC-specific treatment record and no competing cancer code, added back as cases. 
no_other_cancer_ids = set(selfrep_ids) - controls_with_cancer

ids_str_tx = ','.join(map(str, sorted(no_other_cancer_ids)))
q_tx_candidates = f"""
SELECT DISTINCT po.person_id
FROM `{CDR}.procedure_occurrence` po
JOIN `{CDR}.concept` c ON c.concept_id = po.procedure_concept_id
WHERE po.person_id IN ({ids_str_tx})
  AND ( (LOWER(c.concept_name) LIKE '%hysterectomy%' AND LOWER(c.concept_name) LIKE '%lymphadenectomy%')
     OR LOWER(c.concept_name) LIKE '%uterine tandem%' OR LOWER(c.concept_name) LIKE '%vaginal ovoid%'
     OR LOWER(c.concept_name) LIKE '%brachytherapy%' )
"""
tx_candidates = pandas_gbq.read_gbq(q_tx_candidates, project_id=project_id, dialect='standard', progress_bar_type=None)
new_case_ids = sorted(set(tx_candidates['person_id'].astype(int)))

print(f"Self-reporters with no other cancer diagnosis: {len(no_other_cancer_ids):,}")
print(f"Treatment-identified cases found: {len(new_case_ids)}")

print(f"Adding {len(new_case_ids)} treatment-identified case(s) to the cohort")

to_add = pd.DataFrame({'person_id': new_case_ids})
to_add['EC_case'] = 1
for col in final_cohort.columns:
    if col not in to_add.columns:
        to_add[col] = np.nan

final_cohort = pd.concat([final_cohort, to_add[final_cohort.columns]], ignore_index=True)

print(f"Cases after adding treatment-identified ones: {int((final_cohort['EC_case']==1).sum()):,}")
print(f"Total N: {len(final_cohort):,}")



In [ ]:
# Backfilling ancestry/PCs/sex and a diagnosis-date proxy for the new cases
anc_cols = ['person_id', 'ancestry_pred', 'ancestry_pred_forced', 'dragen_sex_ploidy'] + [f'PC{i}' for i in range(1,17)]
lookup = wgs_female[wgs_female['person_id'].isin(new_case_ids)][anc_cols]

for col in anc_cols[1:]:
    final_cohort.loc[final_cohort['person_id'].isin(new_case_ids), col] = \
        final_cohort.loc[final_cohort['person_id'].isin(new_case_ids), 'person_id'].map(
            lookup.set_index('person_id')[col]
        )

# Diagnosis-date proxy: earliest/latest EC-specific treatment date (staging surgery / intrauterine brachytherapy)
nc = ','.join(map(str, new_case_ids))
q = f"""
SELECT person_id, MIN(dt) AS first_tx_date, MAX(dt) AS last_tx_date, COUNT(*) AS n_tx_records
FROM (
  SELECT po.person_id, po.procedure_date AS dt
  FROM `{CDR}.procedure_occurrence` po JOIN `{CDR}.concept` c ON c.concept_id = po.procedure_concept_id
  WHERE po.person_id IN ({nc})
    AND ( (LOWER(c.concept_name) LIKE '%hysterectomy%' AND LOWER(c.concept_name) LIKE '%lymphadenectomy%')
       OR LOWER(c.concept_name) LIKE '%uterine tandem%' OR LOWER(c.concept_name) LIKE '%vaginal ovoid%'
       OR LOWER(c.concept_name) LIKE '%brachytherapy%' )
)
GROUP BY person_id
"""
tx = pandas_gbq.read_gbq(q, project_id=project_id, dialect='standard', progress_bar_type=None)
tx['first_tx_date'] = pd.to_datetime(tx['first_tx_date'])
tx['last_tx_date']  = pd.to_datetime(tx['last_tx_date'])
tx_map = tx.set_index('person_id')

mask = final_cohort['person_id'].isin(new_case_ids)
final_cohort.loc[mask, 'first_diagnosis_date'] = final_cohort.loc[mask,'person_id'].map(tx_map['first_tx_date'])
final_cohort.loc[mask, 'last_diagnosis_date']  = final_cohort.loc[mask,'person_id'].map(tx_map['last_tx_date'])
final_cohort.loc[mask, 'n_records'] = final_cohort.loc[mask,'person_id'].map(tx_map['n_tx_records'])
final_cohort.loc[mask, 'single_record_flag'] = final_cohort.loc[mask,'n_records'] <= 1
final_cohort.loc[mask, 'hyst_timing'] = 'Treatment-identified (no dx code; staging surgery/brachytherapy)'

check = final_cohort[final_cohort['person_id'].isin(new_case_ids)]
print("Any remaining NaN for the new cases?")
print(check.isna().sum()[check.isna().sum() > 0])


In [ ]:
# Enforcing eligibility checks for the treatment-identified cases, evaluated against the actual final retained cohort
# A failure here should stop execution before saving so these are hard assertions rather than warnings
assert len(set(new_case_ids)) == len(new_case_ids), "Duplicate IDs in new_case_ids"

wgs_female_ids = set(int(x) for x in wgs_female['person_id'])
not_eligible = set(new_case_ids) - wgs_female_ids
assert not not_eligible, f"Treatment-identified case(s) not in the WGS/genetic-female pool: {sorted(not_eligible)}"

treatment_ids_flagged = set(new_case_ids) & flagged_in_cohort
assert not treatment_ids_flagged, f"Treatment-identified case(s) are QC-flagged: {sorted(treatment_ids_flagged)}"
print(f"Confirmed: all {len(new_case_ids)} treatment-identified case(s) are WGS/genetic-female and not QC-flagged.")

# Relatedness check evaluated against the actual retained cohort (not the original wgs_female/kinship_in_cohort pools) 
# A kinship link to someone already excluded for an unrelated reason should not falsely flag here.
retained_ids = set(int(x) for x in final_cohort['person_id'])
kinship_among_retained = kinship_in_cohort[
    kinship_in_cohort['i.s'].isin(retained_ids) & kinship_in_cohort['j.s'].isin(retained_ids)
]
related_pairs_among_retained = kinship_among_retained[kinship_among_retained['kin'] > 0.0884]

treatment_ids_in_related_pair = set(new_case_ids) & (
    set(related_pairs_among_retained['i.s']) | set(related_pairs_among_retained['j.s'])
)
assert not treatment_ids_in_related_pair, (
    f"Treatment-identified case(s) are related (kin > 0.0884) to another RETAINED cohort "
    f"member: {sorted(treatment_ids_in_related_pair)}. Resolve before saving."
)
print(f"Confirmed: none of the {len(new_case_ids)} treatment-identified case(s) is related to another retained cohort member.")

check_rows = final_cohort[final_cohort['person_id'].isin(new_case_ids)]
assert len(check_rows) == len(new_case_ids), f"Expected {len(new_case_ids)} row(s) for treatment-identified cases in final_cohort, found {len(check_rows)}"
assert check_rows['person_id'].duplicated().sum() == 0, "Duplicate treatment-identified case rows in final_cohort"
assert check_rows['first_diagnosis_date'].notna().all(), "One or more treatment-identified cases is missing a diagnosis-date proxy"
print(f"Confirmed: all {len(new_case_ids)} treatment-identified case(s) appear exactly once in final_cohort with a valid diagnosis-date proxy.")

print("\nAll treatment-case eligibility checks passed.")


In [ ]:
# All controls are scanned for a diagnosis of endometrial hyperplasia or endometrial intraepithelial neoplasia (EIN)
# This is a recognised, but not universal, precursor lesion. 
# Non-destructive flag rather than an exclusion
# Main analysis retains these controls, with their exclusion reported separately as a sensitivity analysis.
control_ids = final_cohort.loc[final_cohort['EC_case']==0, 'person_id'].astype(int).tolist()

patterns = ["%endometrial hyperplasia%", "%hyperplasia of endometrium%",
            "%complex atypical hyperplasia%", "%atypical endometrial hyperplasia%",
            "%endometrial intraepithelial neoplasia%"]
like_sql = " OR ".join("LOWER(concept_name) LIKE '" + p + "'" for p in patterns)

concept_query = "SELECT concept_id, concept_name FROM `" + CDR + ".concept` WHERE domain_id='Condition' AND (" + like_sql + ")"
concepts = pandas_gbq.read_gbq(concept_query, project_id=project_id, dialect='standard', progress_bar_type=None)
c_str = ','.join(map(str, concepts['concept_id'].tolist()))

parts = []
for b in chunks(control_ids, 40000):
    s = ','.join(map(str, b))
    q = ("SELECT DISTINCT co.person_id FROM `" + CDR + ".condition_occurrence` co "
         "WHERE co.person_id IN (" + s + ") "
         "AND (co.condition_concept_id IN (" + c_str + ") OR co.condition_source_concept_id IN (" + c_str + "))")
    parts.append(pandas_gbq.read_gbq(q, project_id=project_id, dialect='standard', progress_bar_type=None))
hyperplasia_ctrl = pd.concat(parts, ignore_index=True)['person_id'].astype(int).unique()

final_cohort['hyperplasia_ein_flag'] = final_cohort['person_id'].isin(hyperplasia_ctrl)
print(f"Controls flagged for hyperplasia/EIN: {len(hyperplasia_ctrl):,} / {len(control_ids):,} "
      f"({100*len(hyperplasia_ctrl)/len(control_ids):.2f}%)")


## Final cohort checks, and save

In [ ]:
# VerifyING final cohort counts and asserting data integrity before saving
print("N:", len(final_cohort))
print("Cases:", int((final_cohort['EC_case']==1).sum()))
print("Controls:", int((final_cohort['EC_case']==0).sum()))
print("Duplicate person_ids:", final_cohort['person_id'].duplicated().sum())
print("Missing ancestry_pred:", final_cohort['ancestry_pred'].isna().sum())
print("Missing PC1:", final_cohort['PC1'].isna().sum())

assert final_cohort['person_id'].is_unique, "Duplicate person_id in final_cohort -- do not save"
assert final_cohort['EC_case'].isin([0, 1]).all(), "EC_case contains values other than 0/1"
assert final_cohort['ancestry_pred'].notna().all(), "Missing ancestry_pred -- do not save"
assert final_cohort.loc[final_cohort['EC_case']==1, 'first_diagnosis_date'].notna().all(), \
    "A case is missing first_diagnosis_date -- do not save"

# Checking all 10 genetic PCs used throughout the downstream models, and that they are finite
pc_cols_used = [f'PC{i}' for i in range(1, 11)]
pc_values = final_cohort[pc_cols_used].to_numpy(dtype=float)
assert np.isfinite(pc_values).all(), "Missing or non-finite values in PCs 1-10 -- do not save"
assert final_cohort['dragen_sex_ploidy'].eq('XX').all(), "Non-XX participant(s) present in final_cohort -- do not save"
print(f"Confirmed: PCs 1-10 are all present and finite; all participants are genetically XX.")

print("\nAll pre-save integrity checks passed.")


In [ ]:
# Direct reconciliation of the case-side exclusion cascade
# Confirms exactly where the reported case count comes from, and that the hysterectomy-timing and relatedness/QC exclusion sets do not overlap
case_start = set(ec_hyst['person_id'])
post_hyst = set(ec_after_hyst_exclusion['person_id'])
post_all = set(final_cases['person_id'])

hyst_removed = case_start - post_hyst
other_removed_after_hyst = post_hyst - post_all
overlap = hyst_removed & to_remove

print("Initial eligible cases:", len(case_start))
print("Removed for hysterectomy:", len(hyst_removed))
print("Additional relatedness/QC removals:", len(other_removed_after_hyst))
print("Overlapping exclusion flags (hysterectomy-removed AND in to_remove):", len(overlap))
print("Retained coded cases:", len(post_all))
print("Final cases incl. treatment additions:", len(post_all) + len(new_case_ids))

assert len(case_start) - len(hyst_removed) - len(other_removed_after_hyst) == len(post_all), (
    "Sequential-deduction arithmetic does not reconcile with the actual retained set -- "
    "investigate overlap between exclusion reasons."
)
assert len(overlap) == 0, (
    f"Hysterectomy-timing and relatedness/QC exclusion sets overlap ({len(overlap)} people) -- "
    "the exclusion cascade is not a clean sequential deduction; investigate before reporting counts."
)
print("\nSequential-deduction arithmetic reconciles exactly with the actual retained set.")


In [ ]:
# Saving the final cohort, with export verification
out_path = '/home/jupyter/02_ec_final_cohort_excluded.csv'
final_cohort.to_csv(out_path, index=False)
r = subprocess.run(['gsutil','-u',project_id,'cp',out_path,f'{bucket_output}/02_ec_final_cohort_excluded.csv'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode != 0:
    raise RuntimeError(f"gsutil upload failed (return code {r.returncode}): {r.stderr}")

# Verifying the uploaded file by re-downloading and checking it matches what was just saved
verify_path = '/home/jupyter/_verify_02_ec_final_cohort_excluded.csv'
r_verify = subprocess.run(['gsutil', '-u', project_id, 'cp', f'{bucket_output}/02_ec_final_cohort_excluded.csv', verify_path],
                           capture_output=True, text=True)
if r_verify.returncode != 0:
    raise RuntimeError(f"Could not re-download the just-uploaded file for verification: {r_verify.stderr}")
uploaded = pd.read_csv(verify_path, low_memory=False)
assert len(uploaded) == len(final_cohort), (
    f"Uploaded file row count ({len(uploaded):,}) does not match final_cohort ({len(final_cohort):,}) -- upload may be corrupt or stale"
)
assert set(uploaded['person_id']) == set(final_cohort['person_id']), "Uploaded file's person_ids do not match final_cohort"

print(f"Saved and verified: N={len(final_cohort):,}, cases={int((final_cohort['EC_case']==1).sum()):,}, "
      f"controls={int((final_cohort['EC_case']==0).sum()):,}, "
      f"hyperplasia_ein_flag={int(final_cohort['hyperplasia_ein_flag'].sum()):,}")
